# 16 Reward_Model_And_RL_Intro

## Problem

为什么 SFT 之后还常常需要奖励模型和偏好优化？“回答得像人喜欢的样子”与“只是语言通顺”有什么不同？

## Dependencies

建议先理解 SFT、偏好数据、token-level loss 和基本生成流程。


## Module_Goals

- 理解奖励模型、偏好数据、RLHF 的基本角色
- 理解为什么“更符合偏好”不是单纯的 next-token prediction
- 理解 reward model、PPO、DPO 在流程中的相对位置
- 能写一个最小 pairwise preference loss 和一个最小 reward scoring 示例

## Scope_And_Approach

这个 notebook 重点回答一个工程问题：模型已经会说话以后，怎样继续把它往“更有帮助、更符合偏好、更可控”的方向推。这里先建立 RLHF 的基础直觉，再用最小示例说明 reward model 在优化链路里的作用。


## Context_And_Motivation

SFT 解决的是“让模型更像一个会配合指令的助手”，但它不保证模型在多个都通顺的回答中，总能稳定选出更符合人类偏好的那个。一个回答可能：

- 语法上完全没问题，但太绕。
- 内容看着合理，但没有直答问题。
- 格式正确，但帮助性不足。
- 立场过硬或风险控制不稳定。

于是训练目标就不再只是“预测最可能的下一个 token”，而是“在多个候选回答中，更稳定地偏向人类更喜欢的那个”。常见的链路是：

1. 收集 preference pairs 或 ranking data。
2. 训练 reward model，让它学会给回答打分。
3. 再通过 RLHF 或直接偏好优化方法，把 policy 往更高偏好方向更新。


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 一个最小 pairwise preference 示例
chosen_score = 2.4
rejected_score = 1.1

# Bradley-Terry 风格：希望 chosen 的分数高于 rejected
pairwise_loss = -np.log(1 / (1 + np.exp(-(chosen_score - rejected_score))))

print('chosen_score =', chosen_score)
print('rejected_score =', rejected_score)
print('pairwise_preference_loss =', pairwise_loss)


In [ ]:
# 一个最小 reward model 示例：把回答特征映射成分数
# 三个特征仅用于说明：directness, completeness, safety
answers = np.array([
    [0.9, 0.7, 0.8],
    [0.4, 0.8, 0.6],
    [0.7, 0.5, 0.9],
])
reward_w = np.array([0.5, 0.3, 0.6])
rewards = answers @ reward_w

print('answers =')
print(answers)
print('reward scores =', rewards)
print('best answer index =', rewards.argmax())


## RLHF_Positioning

把 reward model 放进完整链路里看，会更清楚：

- **Pretraining**：建立广义语言能力。
- **SFT**：把模型推向更稳定的指令跟随分布。
- **Reward_Model**：把人类偏好压成可优化的分数信号。
- **RLHF / Preference_Optimization**：让 policy 按 reward 或 preference 进一步更新。

这里 reward model 不是替代主模型，而是给主模型提供“哪个回答更值得保留”的训练信号。

## Math

reward model 常见的 pairwise 训练目标可以写成：

$$L_{RM} = -\log \sigma(r(x, y_{chosen}) - r(x, y_{rejected}))$$

它的意思很直接：希望对同一输入 `x`，chosen 回答的 reward 高于 rejected 回答。

如果后续走 PPO 路线，reward model 产出的分数会变成策略优化时的一部分回报信号；如果走 DPO 这类直接偏好优化路线，则更像是绕过显式 RL rollout/value learning，直接在 preference pair 上更新 policy。


In [ ]:
# 一个最小 DPO 风格直觉：比较 policy 与 reference model 的相对偏好差
policy_logprob_chosen = -1.2
policy_logprob_rejected = -2.0
ref_logprob_chosen = -1.5
ref_logprob_rejected = -1.8
beta = 0.1

policy_gap = policy_logprob_chosen - policy_logprob_rejected
ref_gap = ref_logprob_chosen - ref_logprob_rejected
dpo_margin = beta * (policy_gap - ref_gap)
dpo_loss = -np.log(1 / (1 + np.exp(dpo_margin)))

print('policy_gap =', policy_gap)
print('ref_gap =', ref_gap)
print('dpo_margin =', dpo_margin)
print('dpo_loss =', dpo_loss)


## Trade_Offs_And_Risk_Points

- 把 reward model 理解成“判断真伪”的模型。它更准确地说是在拟合偏好信号，而不是真理裁判。
- 以为 RLHF 或 DPO 是让模型突然学会知识。它们更多是在已有能力之上调整行为方向。
- 忽略偏好数据质量。偏好优化是否有效，很大程度取决于 preference signal 本身。
- reward model 如果只学到表面风格，可能会把模型往“高分模板化输出”推，而不是更真实地提高帮助性。

## Checkpoints

- 解释为什么同一个问题会存在多个“语言上都通顺”但偏好不同的答案。
- 解释 reward model 在 SFT 和 RLHF 之间起什么桥梁作用。
- 解释 DPO 相比完整 RL 路线为什么常被认为更直接。
- 说明为什么偏好优化不应被误解成知识预训练的替代品。

## Summary

奖励模型和偏好优化让大模型不仅会“说得像”，还更努力去“说得更符合目标偏好”。reward model 提供可学习的偏好分数，RLHF、PPO、DPO 等方法则负责把这些偏好真正压回 policy 本身。

## Next_Module

下一步会把 reward model 放回更完整的多阶段训练链路里，看一个 DeepSeek-like 模型从 pretraining 到 RLHF / PPO / GRPO 是怎样逐步迭代出来的。
